# Station Analysis

In [3]:
import os
import pandas as pd
outputpath = 'S:/pools/t/T-IDP-Projekte-u-Vorlesungen/Meteoblue/QRF/Data/FINAL/Training/'
palmfilename = 'mb4'
runs = [1, 2, 3, 4]
runfolders = [f'{palmfilename}_run{run}' for run in runs]

In [4]:
def make_lists(x):
    """ Comes in the form [val1 val2 val3] and must be converted to a list with floats"""
    vals = x.replace('[','').replace(']','')
    vals = ' '.join(vals.split())
    vals = vals.split(' ')

    return [float(val) for val in vals]

def extract_prediction(x: pd.DataFrame):
    """ Extracts the prediction and upper / lower confidence intervals from the prediction list"""
    preds = []
    uCI = []
    lCI = []
    for row in range(len(x)):
        lCI.append(x['Prediction'].iloc[row][0])
        preds.append(x['Prediction'].iloc[row][1])
        uCI.append(x['Prediction'].iloc[row][2])
    return preds, uCI, lCI

In [8]:
statistics = {}
for runfolder in runfolders:
    outputfile = os.path.join(outputpath, runfolder, f'{runfolder}.csv')
    trainingfile = os.path.join(outputpath, runfolder, f'{runfolder}_trainingset.csv')
    outputfile = pd.read_csv(outputfile)
    trainingfile = pd.read_csv(trainingfile)
    outputfile['Prediction'] = outputfile['Prediction'].apply(make_lists)
    outputfile['Predicted Temperature'], outputfile['Upper Boundary'], outputfile['Lower Boundary'] = extract_prediction(outputfile)
    outputfile['Error'] = outputfile['True Temperature'] - outputfile['Predicted Temperature']
    stats = {}
    for station_id in outputfile['stationid'].unique():
        station_data = outputfile[outputfile['stationid'] == station_id]
        rmse = ((station_data['Error'] ** 2).mean()) ** 0.5
        mean_error = station_data['Error'].mean()
        max_error = station_data['Error'].max()
        min_error = station_data['Error'].min()
        median_error = station_data['Error'].median()
        std_deviation = station_data['Error'].std()
        stats = {'RMSE': [rmse], 'Mean Error': [mean_error], 'Max Error': [max_error], 
                 'Min Error': [min_error], 'Median Error': [median_error], 'Standard Deviation': [std_deviation]}
        if station_id not in statistics.keys():
            statistics[station_id] = stats
        else:
            for key in statistics[station_id].keys():
                statistics[station_id][key].append(stats[key][0])
        

In [9]:
run_comp = pd.DataFrame(statistics)